# Euler's Method: Walking a Solution One Step at a Time

**Time:** about 15 to 20 minutes.
**You need:** nothing but this page. New to notebooks? Do `00-start-here.ipynb` first.

Imagine driving a car whose odometer is broken but whose speedometer works fine. You cannot read
off how far you have gone. What you can do is glance at the speedometer, assume the speed holds
for the next few seconds, and add up those short stretches.

That is the whole idea behind Euler's method. A differential equation

$$y'=f(t,y)$$

is a speedometer. It never tells you the value of $y$, but at any point it tells you the slope
there. So stand at your starting point, read the slope, walk a short straight distance in that
direction, and then read the slope again from wherever you landed.

> **The main idea:** replace the curve you cannot find with a chain of short straight steps, each
> one aimed along the slope the equation demands where the step begins.

Everything below is a consequence of that one sentence, including the ways it can go wrong.

Each section will ask you to **predict** before you run anything. 

Commit to an answer first, on paper or
out loud.

Being wrong and seeing why is the point!

## Run this first

The two cells below set up the drawing tools. You never need to read or change them. Run them
once each (**Shift + Enter**) and give them a moment on the first run.

In [ ]:
%pip install -q ipywidgets

In [3]:
import numpy as np
import matplotlib.pyplot as plt


def euler(f, t0, y0, h, t_end):
    'Euler steps of size h from (t0, y0) out to t_end. Returns (ts, ys).'
    n = max(int(round((t_end - t0) / h)), 1)
    ts, ys = [t0], [y0]
    t, y = t0, y0
    for _ in range(n):
        y = y + h * f(t, y)
        t = t + h
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)


def rk4(f, t0, y0, t_end, steps=2000):
    'A very accurate reference solution, for comparison only. Returns (ts, ys).'
    h = (t_end - t0) / steps
    ts, ys = [t0], [y0]
    t, y = t0, y0
    for _ in range(steps):
        k1 = f(t, y)
        k2 = f(t + h/2, y + h*k1/2)
        k3 = f(t + h/2, y + h*k2/2)
        k4 = f(t + h, y + h*k3)
        y = y + h*(k1 + 2*k2 + 2*k3 + k4)/6
        t = t + h
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)


def field(ax, f, t_range, y_range, density=19):
    'Draw the slope field of y prime = f(t, y) onto an existing axis.'
    T, Y = np.meshgrid(np.linspace(*t_range, density),
                       np.linspace(*y_range, density))
    M = f(T, Y)
    scale = np.sqrt(1 + M**2)
    ax.quiver(T, Y, 1/scale, M/scale, angles='xy', pivot='middle',
              color='0.55', headlength=0, headwidth=0, headaxislength=0)


def show_euler(f, t0, y0, h, t_end, exact=None, y_range=None,
               title=None, show_field=True, steps_shown=True):
    'Plot the Euler polygon against an accurate solution, optionally over a slope field.'
    ts, ys = euler(f, t0, y0, h, t_end)

    if exact is not None:
        rt = np.linspace(t0, t_end, 400)
        ry = exact(rt)
        ref_label = 'true solution'
    else:
        rt, ry = rk4(f, t0, y0, t_end)
        ref_label = 'accurate solution'

    if y_range is None:
        lo = min(np.nanmin(ys), np.nanmin(ry))
        hi = max(np.nanmax(ys), np.nanmax(ry))
        pad = 0.12 * max(hi - lo, 1e-6)
        y_range = (lo - pad, hi + pad)

    fig, ax = plt.subplots(figsize=(8, 5))
    if show_field:
        field(ax, f, (t0, t_end), y_range)

    ax.plot(rt, ry, color='C3', linestyle='--', linewidth=2.5, label=ref_label)
    ax.plot(ts, ys, color='C0', linestyle='-', linewidth=2,
            marker='o' if steps_shown else None, markersize=5,
            label=f'Euler, h = {h:g}')

    ax.set_xlim(t0, t_end)
    ax.set_ylim(y_range)
    ax.set_xlabel('t')
    ax.set_ylabel('y')
    ax.set_title(title or f'Euler with h = {h:g}')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper left', framealpha=0.9)

    plt.show()
    plt.close(fig)


print('Ready.')

Ready.


## 1. One step, by hand

Start with the simplest equation in the course:

$$y'=y, \qquad y(0)=1.$$

Read it as a sentence: *the slope equals the height*. You are standing at height $1$, so the
equation says your slope right now is $1$.

Take one step of size $h=1$. Walk from $t=0$ to $t=1$ along a straight line of slope $1$.

Before running anything, work these out:

1. What height do you land at?
2. The true solution is $y=e^{t}$, so the real height at $t=1$ is about $2.72$. Are you above it
   or below it?
3. Why did that happen? Look at the shape of $e^t$ and think about where your straight line sat
   relative to the curve.

In [ ]:
show_euler(lambda t, y: y, t0=0.0, y0=1.0, h=1.0, t_end=1.0,
           exact=np.exp, y_range=(0.5, 3.2),
           title="y' = y, y(0) = 1: a single Euler step of size h = 1")

### What the picture shows

The blue line is your step. It leaves the starting point aimed exactly along the slope field, and
for a moment it is a perfect description of the solution. Then the solution curve bends upward and
your straight line does not.

You landed at $y=2$. The truth is $e \approx 2.718$, so you are short by about $0.72$.

<details>
<summary>Check yourself after you have written something</summary>

You land at $1 + 1 \cdot 1 = 2$, which is <strong>below</strong> the true value. The curve $y=e^t$
is concave up, meaning it curves away above every one of its tangent lines. Euler walks along the
tangent line at the start of the step, so on a concave up solution it always falls short.
</details>

## 2. Chain the steps together

One step is not a method. The method is what happens when you land, read the slope again from the
new point, and step once more.

Same equation, same start, still $h=1$, but now walk all the way out to $t=3$. That is three
steps. Do the arithmetic in your head before you run the cell, because it is easy:

$$y_{0}=1, \qquad y_{n+1}=y_{n}+h\,f(t_{n},y_{n})=y_{n}+y_{n}.$$

Each step just doubles the height.

Predict:

1. What are the four values $y_0, y_1, y_2, y_3$?
2. The true answer is $e^{3}\approx 20.09$. How badly off are you?
3. Does the gap grow, shrink, or hold steady as you take more steps?

In [ ]:
show_euler(lambda t, y: y, t0=0.0, y0=1.0, h=1.0, t_end=3.0,
           exact=np.exp, y_range=(0, 22),
           title="y' = y, y(0) = 1: three steps of size h = 1")

### Errors compound

You got $1, 2, 4, 8$. The truth at $t=3$ is about $20.09$, so you finished at less than half the
right answer.

The reason matters more than the number. Each step starts from wherever the previous step left
you, and by step two you are already standing in the wrong place, so you read the slope from the
wrong place too. The error at each step is small, but the error you carry forward is not, because
every later step inherits it.

Notice also that the polygon is still doing something recognizable. It rises, it accelerates, it
has the right qualitative story. It is only the numbers that are bad.

<details>
<summary>Check yourself</summary>

$y_0=1$, $y_1=2$, $y_2=4$, $y_3=8$. The gap grows: after one step it is about $0.72$, after three
it is about $12.1$. The gap widens because the solution is growing and the errors compound
alongside it.
</details>

## 3. Shrink the step

A step of size $1$ on a curve that doubles every unit of time was never going to work. So make the
steps smaller.

Before you touch the slider, predict what will happen at the two extremes:

- At $h=1.0$ you already know the answer is $8$ against a true $20.09$.
- At $h=0.05$ you take sixty steps instead of three. Will the polygon land on the curve exactly,
  or just close? Say which, and say why.

Now run the cell and drag $h$ all the way to both ends of its range on purpose. The ends are where
the information is. The error at $t=3$ is printed in the title.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_step_size(h=1.0):
    f = lambda t, y: y
    ts, ys = euler(f, 0.0, 1.0, h, 3.0)
    err = abs(np.exp(3) - ys[-1])
    show_euler(f, t0=0.0, y0=1.0, h=h, t_end=3.0, exact=np.exp, y_range=(0, 22),
               steps_shown=(h >= 0.2),
               title=f"y' = y with h = {h:.2f}:  {len(ts)-1} steps, "
                     f"final y = {ys[-1]:.3f}, error = {err:.3f}")

interact(
    explore_step_size,
    h=FloatSlider(value=1.0, min=0.05, max=1.0, step=0.05,
                  description='h', continuous_update=False)
);

### What changed, and what stayed the same?

The polygon tightens onto the curve as $h$ shrinks, and the corners smooth out until you cannot
see them any more. That part is probably what you expected.

Here is the part worth noticing. The polygon is **always below the curve**, at every step size you
can select. Shrinking $h$ does not make Euler start guessing high sometimes and low other times.
It shrinks a bias that was there the whole time and never removes it. That is a property of the
method meeting a concave up solution, not bad luck.

**In your notes,** fill these in from the slider:

- At $h=1.00$, the error at $t=3$ is about ________.
- At $h=0.50$, about ________.
- At $h=0.05$, about ________.

Cutting $h$ by a factor of twenty took the error from about $12$ to about $1.4$. Better, and also
worse than you might have hoped for twenty times the work. The next section pins down the trade.

## 4. Halve the step, and the error does what?

You now have a method with a dial on it. The useful question about any such method is what you buy
per unit of extra work.

Predict before running. Each row below halves $h$, which doubles the number of steps:

- If halving $h$ **halves** the error, the last column will settle near $2$.
- If halving $h$ **quarters** the error, it will settle near $4$.

Write down which one you expect. Then run the cell.

In [ ]:
f = lambda t, y: y
true = np.exp(3)

print(f"y' = y, y(0) = 1, marching to t = 3.   True value: e^3 = {true:.6f}")
print()
print(f"{'h':>10} {'steps':>7} {'Euler y(3)':>12} {'error':>10} {'previous/this':>15}")
print('-' * 58)

prev = None
for h in [0.5, 0.25, 0.125, 0.0625, 0.03125]:
    ts, ys = euler(f, 0.0, 1.0, h, 3.0)
    err = abs(true - ys[-1])
    ratio = f'{prev/err:.3f}' if prev is not None else '-'
    print(f'{h:>10.5f} {len(ts)-1:>7} {ys[-1]:>12.4f} {err:>10.4f} {ratio:>15}')
    prev = err

### First order, and what that costs you

The ratios run $1.57$, $1.73$, $1.85$, $1.92$. They are climbing toward $2$ without quite getting
there, which is what "approaches $2$ as $h$ gets small" looks like in an actual table rather than
in a theorem.

So halving the step size buys you roughly half the error. Euler's method is **first order**: the
error at a fixed time shrinks in proportion to $h$.

That is a bad exchange rate. To get one more decimal place of accuracy you need ten times as many
steps. To get three more, a thousand times as many. This is exactly why the next two sections of
the textbook exist. The improved Euler and Runge Kutta methods do more arithmetic per step in
order to buy a much better exponent, and once you have seen this table you know what they are
buying.

**In your notes:** you saw the error at $h=0.03125$ was about $0.90$. Roughly what $h$ would you
need to get the error under $0.01$? About how many steps is that?

<details>
<summary>Check yourself</summary>

Error is proportional to $h$, so to divide the error by $90$ you divide $h$ by about $90$, giving
$h \approx 0.00035$. Marching to $t=3$ then takes roughly $8600$ steps, to compute one number you
already know is $e^3$. First order accuracy is genuinely expensive.
</details>

## 5. Small steps are not always small enough

Everything so far said the same comforting thing: bigger $h$ is worse, smaller $h$ is better, and
the errors behave themselves in between. Now watch that comfort break.

Take a system that settles down fast. A hot cup of coffee in a cold room, a capacitor discharging,
a drug clearing quickly from the bloodstream:

$$y'=-10y, \qquad y(0)=1.$$

Read it as a sentence: *the rate of change is ten times the current amount, pointing back toward
zero*. The true solution is $y=e^{-10t}$. It drops toward zero and stays there, and it is never
negative, because starting positive and heading toward zero gives it nowhere else to go.

Before you touch the slider, predict what Euler will do with $h=0.3$:

1. The true solution at $t=2$ is about $2\times10^{-9}$, which is zero for any practical purpose.
   What do you expect Euler to report?
2. Could Euler ever return a negative value here, given that the true solution never is?

Now drag $h$ slowly upward from the left end. Watch what happens as you pass $0.2$.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_stability(h=0.05):
    f = lambda t, y: -10 * y
    ts, ys = euler(f, 0.0, 1.0, h, 2.0)
    peak = np.max(np.abs(ys))
    verdict = 'decays' if peak <= 1.0001 else 'BLOWING UP'
    show_euler(f, t0=0.0, y0=1.0, h=h, t_end=2.0,
               exact=lambda t: np.exp(-10*t), show_field=False,
               title=f"y' = -10y with h = {h:.2f}:  growth factor "
                     f"1 - 10h = {1-10*h:.2f}, largest |y| = {peak:.3g} ({verdict})")

interact(
    explore_stability,
    h=FloatSlider(value=0.05, min=0.02, max=0.35, step=0.01,
                  description='h', continuous_update=False)
);

### The knife edge at $h=0.2$

Push the slider to both extremes and find the boundary. Below $h=0.2$ the polygon decays like the
true solution. Above it, Euler produces a zigzag that grows without bound, alternating sign at
every step, while the thing it claims to be approximating is quietly sitting at zero.

You can see exactly why from one line of algebra. For $y'=-10y$ the Euler step is

$$y_{n+1}=y_{n}+h(-10y_{n})=(1-10h)\,y_{n},$$

so each step multiplies by the fixed number $1-10h$. That is the growth factor printed in the
title. The steps shrink only when $|1-10h|<1$, which is the same as $0<h<0.2$. At $h=0.2$ exactly
the factor is $-1$ and the values bounce between $1$ and $-1$ forever without ever decaying. Past
that, the factor is bigger than $1$ in size and every step multiplies the error again.

This failure is a different animal from the one in sections 2 through 4. That was **accuracy**,
where the answer is off by an amount you can shrink. This is **stability**, where the answer is
not merely inaccurate but qualitatively wrong: it grows when the truth decays, and it goes negative
when the truth cannot.

**In your notes:** suppose the equation were $y'=-100y$ instead. Where would the stability
boundary sit, and what does that tell you about equations that settle down very fast?

<details>
<summary>Check yourself</summary>

The factor becomes $1-100h$, so stability needs $|1-100h|<1$, which is $0 < h < 0.02$. The faster
the system settles, the smaller a step Euler is forced to take, even though the interesting
behavior is over almost immediately. Equations with this character are called <strong>stiff</strong>,
and they are the reason a whole family of numerical methods exists that Euler does not belong to.
</details>

## 6. Final challenge: read this one cold

No scaffolding here. Work it on paper first, then run the cell.

$$y'=y^{2}, \qquad y(0)=1.$$

This one is separable, so you can actually solve it. Do that now, before reading on.

Then answer:

1. What is the solution $y(t)$?
2. What happens to it as $t\to 1^{-}$?
3. For what values of $t$ does a solution to this initial value problem exist at all?
4. Euler's method never divides by zero and never fails here. So what do you think it will print
   at $t=1.5$?

Commit to answers. Then run the cell.

In [ ]:
f = lambda t, y: y**2
ts, ys = euler(f, 0.0, 1.0, 0.1, 1.5)

print("y' = y^2,  y(0) = 1,  Euler with h = 0.1")
print('True solution: y = 1/(1-t),  which exists only for t < 1')
print()
print(f"{'t':>6} {'Euler says':>14} {'truth':>14}")
print('-' * 38)
for t, y in zip(ts, ys):
    truth = f'{1/(1-t):.4f}' if t < 1.0 - 1e-9 else 'does not exist'
    print(f'{t:>6.1f} {y:>14.4f} {truth:>14}')

In [ ]:
show_euler(lambda t, y: y**2, t0=0.0, y0=1.0, h=0.1, t_end=0.9,
           exact=lambda t: 1/(1-t), y_range=(0, 11),
           title="y' = y^2, y(0) = 1: Euler with h = 0.1, stopped safely at t = 0.9")

### The number that means nothing

Look hard at the row for $t=1.0$. Euler reports $6.13$.

That is not a wild value. It is not infinity, not negative, not an error message. It is a
perfectly ordinary looking number sitting one row below a place where the solution has already
escaped to infinity. If this table came out of a spreadsheet you built for homework, nothing on
the screen would tell you that everything from that row down is fiction.

By $t=1.5$ Euler has worked itself up to $16250$, still confidently, still with no complaint.

The method has no way to know. Euler only ever does one thing, which is to read a slope and take a
step, and that operation is perfectly well defined at every point it visits. Nothing in the
arithmetic asks whether a solution exists out there to be approximated.

<details>
<summary>Check yourself</summary>

Separating gives $-1/y = t + C$, and $y(0)=1$ gives $C=-1$, so $y = 1/(1-t)$. As $t \to 1^-$ the
solution blows up to infinity, so the initial value problem has a solution only on $t<1$. Euler
prints $6.13$ at $t=1$ and $16250$ at $t=1.5$, and every one of those numbers is meaningless.

The plot deliberately stops at $t=0.9$. Even there, safely inside the interval where a solution
exists, look at the gap: Euler says $4.29$ where the truth is $10$. Steep slopes are where a fixed
step size hurts most, and it was already failing badly before it started inventing things.
</details>

## Wrap-up

Euler's method is one idea repeated: read the slope where you are standing, take a short straight
step along it, repeat. You can run it on any first order equation, including every equation in
this course that has no closed form solution. That is the whole reason it is worth knowing.

You also found its three limits, and they are different from each other:

Accuracy is the mild one. The polygon lags the true curve, the lag shrinks in proportion to $h$,
and you can pay for as much accuracy as you want at a poor exchange rate.

Stability is the sharp one. Past a threshold in $h$ the answer stops being merely inaccurate and
starts being the wrong shape entirely, growing where the truth decays.

Existence is the invisible one. The method returns numbers whether or not there is a solution for
those numbers to approximate, and it looks exactly the same either way.

When you do the spreadsheet problems for this section, you will be running the recurrence

$$y_{n+1}=y_{n}+h\,f(t_{n},y_{n})$$

down a column of cells. That column is the picture you have been dragging sliders through, and a
row of numbers cannot tell you which of the three situations above you are in. Knowing what the
equation does qualitatively is what tells you whether to believe the column.

### Exit question

In two or three sentences, explain this in your own words:

> Euler's method always returns an answer, and nothing inside the method checks whether that
> answer means anything.

Name one specific thing you would look at, before or after running it, to decide whether to trust
the numbers you got.